# Milestone 2: bounded quartet training-fit check

Can the unchanged model fit a small, balanced training set with repeated exposure?
Use one arrangement, all shape orders and distinct-color assignments, and all four
circle/square sizes: **144 families, 576 images, 1,728 questions**.

Train fresh seed-0 weights for **2,160 updates / 40 passes**, R=2, float32,
batch 32 and unchanged AdamW. Evaluate training fit only. No validation/test
inference, automatic extension, best-checkpoint selection, or training resume.

Enable GPU and internet. Attach `milestone2_matched_size_training_artifacts.zip`;
only its pinned corpus is read. Select a committed `REPO_REF` after pushing.
Use fresh checkout/output directories and preserve Kaggle's installed PyTorch.
See `docs/milestones/milestone2_quartet_fit.md`.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # Commit SHA preferred; must contain this implementation.
REPO_DIR = "/kaggle/working/multi-modal-loop-quartet-fit"
RUN_ROOT = "/kaggle/working/milestone2_quartet_fit"
REFERENCE_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_matched_size_training_artifacts.zip"
)


## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Derive the balanced training subset

Verify the pinned source manifest, choose the first complete arrangement without
using predictions, and record quartet membership, balance, and exact exposures.
Reference weights are never loaded. Existing held-out origins remain reserved.


In [ ]:
import multimodal_loop.eval.kaggle_quartet_fit as helpers
from multimodal_loop.eval.kaggle_quartet_fit import (
    archive_quartet_fit,
    prepare_quartet_fit,
    run_quartet_fit,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_quartet_fit(REPO_DIR, RUN_ROOT, REFERENCE_SOURCE)


## Train and assess the final checkpoint

Run exactly 40 complete passes, monitoring training fit before training and after
every pass. Final frozen diagnostics cover only the same 1,728 training questions.


In [ ]:
report = run_quartet_fit(run)


## Inspect training fit

Success requires ≥99% accuracy for each queried shape and ≥95% of the 144 families
retaining both circle/square answers correctly across all four sizes. These are
training-fit criteria, not milestone or generalization gates. Failure applies to
this fixed protocol; it does not establish architectural impossibility.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

print("Training fit:", {k: report["training_fit"][k] for k in ("total", "accuracy", "loss")})
print("Per shape:", report["training_fit"]["breakdowns"]["shape"])
print("Families:", {k: report["families"][k] for k in ("total", "correct", "fraction")})
print(json.dumps(report["assessment"], indent=2))
for condition, metrics in report["conditions"].items():
    print(condition, "Circle/square pair:", metrics["circle_square_pair"])
for relation, group in report["relative_size"].items():
    print(relation, "Selections:", group["shape_predictions"])
history = json.loads((run.root / "training/metrics.json").read_text())
for row in history:
    print(row["completed_steps"], row["training_fit"])
display(HTML((run.root / "diagnosis/inspection.html").read_text()))


## Retain artifacts

Download `milestone2_quartet_fit_artifacts.zip` for review. It contains the subset
manifest with source provenance, checkpoint, history, exposure counts, final
predictions, family diagnostics, inspection preview, protocol, hashes and logs.
A pass establishes training fit only; it does not complete Milestone 2.


In [ ]:
archive = archive_quartet_fit(run)
print("Training-fit archive:", archive)
display(FileLink(str(archive)))
